# This script builds a local SQL data warehouse by importing and standardizing market and telemetry CSV data into a structured SQLite database. It then uses advanced SQL window functions to calculate the time elapsed between magistrate inspections, enriching the raw data for further predictive analysis

In [ ]:
import sqlite3
import pandas as pd
import warnings
import os

warnings.filterwarnings('ignore')

def build_data_warehouse():
    print(">>> INITIALIZING PHASE 1.5: PCCMD SQL DATA WAREHOUSE...")

    
    data_folder = os.path.join('..', 'Data')
    processed_folder = os.path.join(data_folder, 'Processed')
    
    # Ensure the database is saved in the Data folder, NOT the Notebooks folder
    db_path = os.path.join(data_folder, 'PCCMD_Warehouse_New2.db')

    # 1. Create a connection to the local SQL Database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 2. LOAD RAW DATA INTO SQL 
    print(">>> Loading CSVs and Standardizing Date Formats...")
    
    try:
        # Using os.path.join for OS-agnostic path building and fixing the filename typo
        market_file = os.path.join(processed_folder, 'Master_Market_Fact.csv')
        telemetry_file = os.path.join(processed_folder, 'Magistrate_Telemetry_Log_New.csv')
        
        market_df = pd.read_csv(market_file)
        telemetry_df = pd.read_csv(telemetry_file)
        
        # --- FIX: Convert '1/20/2026' to '2026-01-20' for SQLite Compatibility ---
        telemetry_df['Date'] = pd.to_datetime(telemetry_df['Date']).dt.strftime('%Y-%m-%d')
        # Also ensure Market dates are standardized if they follow the same pattern
        market_df['Date'] = pd.to_datetime(market_df['Date']).dt.strftime('%Y-%m-%d')
        
        # Write the dataframes to SQL tables
        market_df.to_sql('Fact_Market', conn, if_exists='replace', index=False)
        telemetry_df.to_sql('Fact_Inspections_Raw', conn, if_exists='replace', index=False)
        print("    [+] Raw tables loaded and dates formatted successfully.")
        
    except Exception as e:
        print(f"!!! Error loading CSVs: {e}")
        return

    # 3. BUILD DIMENSION TABLE (Dim_Magistrate)
    print(">>> Building Dim_Magistrate table...")
    cursor.execute("DROP TABLE IF EXISTS Dim_Magistrate;")
    cursor.execute("""
        CREATE TABLE Dim_Magistrate AS
        SELECT DISTINCT Magistrate_ID, City 
        FROM Fact_Inspections_Raw;
    """)

    # 4. WINDOW FUNCTIONS
    print(">>> Executing SQL Window Functions to find minutes between the inspections")
    cursor.execute("DROP TABLE IF EXISTS Fact_Inspections_Enriched;")
    
    advanced_sql_query = """
        CREATE TABLE Fact_Inspections_Enriched AS
        SELECT 
            *,
            ROUND(
                (JULIANDAY(Date || ' ' || Time) -      -- || is the concatenation operator         
                 JULIANDAY(LAG(Date || ' ' || Time) OVER (
                    PARTITION BY Magistrate_ID, Date 
                    ORDER BY Time
                 ))) * 24 * 60, 2        -- Converts days to minutes
            ) AS Minutes_Since_Last_Inspection
        FROM Fact_Inspections_Raw;
    """
    cursor.execute(advanced_sql_query)
    
    # Commit changes
    conn.commit()
    print("    [+] Window functions executed successfully.")
    
    # 5. VERIFY THE RESULTS
    print("\n>>> VERIFYING SQL WAREHOUSE INTEGRITY...")
    
    check_query = """
        SELECT Magistrate_ID, Date, Time, Minutes_Since_Last_Inspection 
        FROM Fact_Inspections_Enriched 
        WHERE Minutes_Since_Last_Inspection IS NOT NULL 
        LIMIT 10;
    """
    test_df = pd.read_sql_query(check_query, conn)
    
    if test_df.empty:
        print("!!! WARNING: Verification failed. Column is still empty. Check Time format (HH:MM:SS).")
    else:
        print("\nSample Output (Success!):")
        print(test_df)
    
    conn.close()
    
    print("\n" + "="*60)
    print(f"SUCCESS: Phase 1.5 Complete. Database saved at: {db_path}")
    print("="*60)

if __name__ == "__main__":
    build_data_warehouse()

>>> INITIALIZING PHASE 1.5: PCCMD SQL DATA WAREHOUSE...
>>> Loading CSVs and Standardizing Date Formats...
    [+] Raw tables loaded and dates formatted successfully.
>>> Building Dim_Magistrate table...
>>> Executing SQL Window Functions for Telemetry Enrichment...
    [+] Window functions executed successfully.

>>> VERIFYING SQL WAREHOUSE INTEGRITY...

Sample Output (Success!):
  Magistrate_ID        Date      Time  Minutes_Since_Last_Inspection
0       MAG-100  2026-01-20  11:32:00                           30.0
1       MAG-100  2026-01-20  12:11:00                           39.0
2       MAG-100  2026-01-20  12:50:00                           39.0
3       MAG-100  2026-01-20  13:31:00                           41.0
4       MAG-100  2026-01-20  14:09:00                           38.0
5       MAG-100  2026-01-20  14:26:00                           17.0
6       MAG-100  2026-01-21  11:09:00                           45.0
7       MAG-100  2026-01-21  11:42:00                           